In [ ]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col, current_date, lit, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================
# ADLS OAuth Configuration
# Replace YOUR_CLIENT_SECRET
# ==========================
for acct in ["adlsdevvbsrc01","adlsdevvbstd01"]:
    spark.conf.set(f"fs.azure.account.auth.type.{acct}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{acct}.dfs.core.windows.net",
                   "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{acct}.dfs.core.windows.net",
                   "2c5d85d1-759d-4e41-9703-905b120f74f5")
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{acct}.dfs.core.windows.net",
                   "YOUR_CLIENT_SECRET")
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{acct}.dfs.core.windows.net",
                   "https://login.microsoftonline.com/7cab1251-58e4-4efb-8936-ae9b0cb98fce/oauth2/token")

# Today's file (or use yesterday by uncommenting)
process_date = datetime.today().strftime("%Y/%m/%d")
# process_date = (datetime.today() - timedelta(days=1)).strftime("%Y/%m/%d")

source_root = "abfss://output@adlsdevvbsrc01.dfs.core.windows.net/fact_scd1"
delta_path = "abfss://output@adlsdevvbstd01.dfs.core.windows.net/order_delta_scd1"

path = f"{source_root}/{process_date}/*.parquet"

raw_df = (spark.read.format("parquet")
          .option("inferSchema","true")
          .load(path))
display(raw_df)

(raw_df.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .save(delta_path))

spark.sql(f"""
CREATE TABLE IF NOT EXISTS order_delta_scd1
USING DELTA
LOCATION '{delta_path}'
""")

# Dynamic latest folder
folders = dbutils.fs.ls(source_root)
latest_year = max(f.name.rstrip("/") for f in folders)

folders = dbutils.fs.ls(f"{source_root}/{latest_year}")
latest_month = max(f.name.rstrip("/") for f in folders)

folders = dbutils.fs.ls(f"{source_root}/{latest_year}/{latest_month}")
latest_day = max(f.name.rstrip("/") for f in folders)

latest_path = f"{source_root}/{latest_year}/{latest_month}/{latest_day}/*.parquet"
file_date = f"{latest_year}-{latest_month}-{latest_day}"

raw_df = spark.read.parquet(latest_path)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled","true")

enriched_df = (raw_df
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date)))

window = Window.partitionBy("ORDER_ID","ORDER_ITEM_ID").orderBy(col("LOAD_TS").desc())

dedup_df = (enriched_df
    .withColumn("rn", row_number().over(window))
    .filter(col("rn")==1)
    .drop("rn"))

table_name="order_delta_scd1"

if not DeltaTable.isDeltaTable(spark, delta_path):
    (dedup_df.write.format("delta")
        .mode("overwrite")
        .option("mergeSchema","true")
        .partitionBy("file_date")
        .save(delta_path))

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION '{delta_path}'
    """)
else:
    deltaTable = DeltaTable.forPath(spark, delta_path)
    (deltaTable.alias("t")
        .merge(dedup_df.alias("s"),
               "t.ORDER_ID=s.ORDER_ID AND t.ORDER_ITEM_ID=s.ORDER_ITEM_ID")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

updated_df = spark.read.format("delta").load(delta_path)

print("Before Merge")
raw_df.show(60, truncate=False)

print("After Merge")
updated_df.show(60, truncate=False)
